In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
info = pd.read_csv('../figS14_reproducibility/full_sample_info.csv',index_col=0)
info

In [ ]:
cuts = np.linspace(0.05,0.95,10)

preseq = []
for animal in info['animal'].unique():
    for col in ['VHH','clone_id']:
        for rep in [['rep1_count'],['rep2_count'],['rep1_count','rep2_count']]:
            file = f'preseq_out/{animal}_{'+'.join([r.split('_')[0] for r in rep])}_{col}_bound.txt'
            pred = pd.read_csv(file,sep='\t')
            pred['animal'] = animal
            pred['col'] = col
            pred['rep'] = '+'.join([r.split('_')[0] for r in rep])
            preseq.append(pred)
            
preseq = pd.concat(preseq,axis=0).reset_index()
preseq

In [ ]:
data = {animal:pd.read_csv(f'../large_files/{animal}_counts_vhh.csv') for animal in info['animal'].unique()}
data['mikka'].head()

In [ ]:
for animal,group in info.groupby('animal'):
    for rep,subgroup in group.groupby('rep'):
        totals = data[animal][subgroup['name'].tolist()].sum(axis=1)
        data[animal][f'rep{rep}_freq'] = totals/totals.sum()
        data[animal][f'rep{rep}_count'] = totals

In [ ]:
for i,row in preseq.iterrows():
    col = row['col']
    fam = data[row['animal']][[col,'rep1_count','rep2_count']].groupby(col).sum()
    
    counts = fam[[f'{r}_count' for r in row['rep'].split('+')]].sum(axis=1)
    preseq.loc[i,'observed'] = (counts>0).sum()
    preseq.loc[i,'total_reads'] = counts.sum()
    
preseq = preseq.rename(columns={'median_estimated_unobs':'median_estimated_total'})
preseq['med_frac_unobs'] = (preseq['median_estimated_total']-preseq['observed'])/preseq['median_estimated_total']
preseq['med_N_unobs'] = (preseq['median_estimated_total']-preseq['observed'])

preseq['type'] = ['Nanobodies' if x == 'VHH' else 'Families' for x in preseq['col']]

In [ ]:
preseq.head()

In [ ]:
plt.figure(figsize=[8.7,13])
letters = 'ABCDEF'
for a,(animal,group) in enumerate(preseq.groupby('animal')):
    ax = plt.subplot(3,2,1+a)
    sns.barplot(data=group,x='type',y='med_frac_unobs',hue='rep',palette='Dark2')
    plt.ylim([0,1])
    plt.title(animal.capitalize(),fontsize=24)

    if a==0:
        plt.ylabel('Fraction Unobserved',fontsize=16)
        plt.legend(title='Replicate')
    else:
        plt.ylabel('')
        plt.legend([],frameon=False)
    ax.text(-0.15, 1.05, letters[a], fontsize=32, transform=ax.transAxes)
    plt.xlabel('')
    plt.xticks(fontsize=16)
    plt.yticks(fontsize=16)

for a,(animal,group) in enumerate(preseq.groupby('animal')):
    ax = plt.subplot(3,2,3+a)
    sns.barplot(data=group,x='type',y='median_estimated_total',hue='rep',palette='Dark2')
    plt.semilogy()
    plt.ylim([3e3,2.3e6])

    if a==0:
        plt.ylabel('Repertoire Size',fontsize=16)
    else:
        plt.ylabel('')
    plt.legend([],frameon=False)
    plt.xlabel('')

    ax.text(-0.15, 1.05, letters[a+2], fontsize=32, transform=ax.transAxes)        
    plt.xticks(fontsize=16)
    plt.yticks(fontsize=16)

for a,(animal,group) in enumerate(preseq.groupby('animal')):
    freq1 = data[animal].groupby('clone_id')['rep1_freq'].sum().values
    freq2 = data[animal].groupby('clone_id')['rep2_freq'].sum().values
    freq1_missing = freq1[freq2==0]
    freq2_missing = freq2[freq1==0]
    
    ax = plt.subplot(3,2,5+a)
    sns.histplot(data=np.concatenate((freq1,freq2),axis=0),bins=np.logspace(-6,-1,30),stat='probability',label='all')
    sns.histplot(data=np.concatenate((freq1_missing,freq2_missing),axis=0),bins=np.logspace(-6,-1,30),stat='probability',label='non-overlapping')
    plt.loglog()

    if a==0:
        plt.xlabel('Family Abundance',fontsize=16,loc='left')
        ax.annotate("", xytext=(0.7, -0.18), xy=(2.25, -0.18), xycoords='axes fraction',
                    arrowprops={'width':1.7,'color':'k'})
        plt.ylabel('Fraction of Families',fontsize=16)
        plt.legend([],frameon=False)
    else:
        plt.xlabel('')
        plt.ylabel('')
        plt.legend()

    ax.text(-0.15, 1.05, letters[a+4], fontsize=32, transform=ax.transAxes)        
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)

plt.subplots_adjust(hspace=0.4,wspace=0.25)
plt.savefig('figS15.png',dpi=300,bbox_inches='tight')